In [ ]:
'''daily
选股'''

In [ ]:
import pandas as pd
from grid_backtest_flexible import (
    BacktestConfig, AnchorConfig, GridSizeConfig,
    LayerConfig, ResetConfig, RiskConfig,
    run_flexible_grid_backtest
)

# 你的分钟数据，至少需要：
# datetime/open/high/low/close/volume
df = pro.stk_mins(ts_code='600000.SH', freq='1min', start_date='2023-08-25 09:00:00', end_date='2023-08-25 19:00:00')

df["datetime"] = pd.to_datetime(df["datetime"])

cfg = BacktestConfig(
    initial_cash=10000,
    units_per_layer=100,
    anchor=AnchorConfig(
        method="VWAP",              # 或 "OPEN15_MID"
        open_minutes=15,
        update_interval_minutes=5,
    ),
    grid=GridSizeConfig(
        method="HYBRID",            # "ATR_MULT" / "PCT" / "NOISE" / "HYBRID"
        atr_mult=0.25,              # ATR_MULT 时用
        pct=0.003,                  # PCT 时用，0.3%
        noise_mult=3.0,             # NOISE 时用
        spread_mult=5.0,
        tick_mult=2.0,
        atr_mult_hybrid=0.25,
        atr_window=20,
        adx_window=14,
        noise_window=20,
    ),
    layers=LayerConfig(
        method="FIXED",             # 或 "HIST30"
        fixed_layers=5,
        hist_lookback_days=30,
        min_layers=3,
        max_layers=12,
    ),
    reset=ResetConfig(
        minutes_same_side=20,
        anchor_shift_grids=1.5,
        adx_threshold=30,
        midday_reanchor=True,
        midday_time="12:00",
        reset_on_adx_trend=True,
        reset_cooldown_minutes=10,
    ),
    risk=RiskConfig(
        fee_rate=0.0005,
        stop_pct_from_anchor=0.03,
        force_close_end_of_day=True,
        session_close_time=None,    # 或比如 "15:55"
        price_field_for_execution="close",
    ),
)

res, stats, final, trade_log, df_feat = run_flexible_grid_backtest(df, cfg)

print(stats)
print(final)
print(pd.DataFrame(trade_log).tail())